# 10 - Enterprise Data Quality & Production Reconciliation

**The 5 Pillars of Enterprise Data Quality & Reconciliation:**
1. **Grain & Contract Assertions:** Asserting PK uniqueness, referential integrity (FK), and schema constraints before publishing.
2. **Automated Null & Duplicate Profiling:** Scanning critical metric columns and business keys across Medallion layers.
3. **Quarantine & Defect Observability:** Quarantining bad records with explicit defect reasons rather than silently dropping rows.
4. **Root-Cause Incident Forensics:** Diagnosing metric discrepancies when downstream dashboards diverge from upstream sources (the "+40% vs +5%" investigation).
5. **Stage-by-Stage & Financial Reconciliation:** Verifying row counts, entity keys, and dollar balances (accounting for voided/reversed payments).


In [ ]:
import sys, pathlib, os, json, time, re
import pandas as pd
from pyspark.sql import functions as F, Window

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

from de_helpers import (
    q, q1, spark_q, assert_grain, tables, columns, explain, display, fs_ls,
    get_spark, get_dbutils, spark_read, plan_string, has_plan_operator
)

spark = get_spark("10-data-quality")
dbutils = get_dbutils()
print("Setup complete. DuckDB warehouse and Spark session initialized.")


---
### Drill 1: A Reusable Data Quality Check Framework

**Business Context:** Does the published layer continue to satisfy schema contracts and data expectations?
* **Assertion Rule:** 
  * `0` violations $\to$ **`PASS`**
  * `violations <= threshold` $\to$ **`WARN`** (acceptable business noise)
  * `violations > threshold` $\to$ **`FAIL`** (blocking data quality incident)
* **Check Scope:** PK uniqueness, referential integrity (FK), null percentages, domain validity, and freshness.


In [ ]:
# YOUR TURN:
# Implement run_check(name, sql, threshold, layer, note) -> 'PASS' | 'WARN' | 'FAIL'
# Append each execution into a scorecard table.


In [ ]:
# SOLUTION - Drill 1: Reusable Check Framework
RESULTS = []

def run_check(name: str, sql: str, threshold: float, layer: str = 'silver', note: str = '') -> str:
    """Evaluates a DQ assertion: 0 -> PASS, <= threshold -> WARN, > threshold -> FAIL."""
    actual = float(q1(sql) or 0)
    if actual == 0:
        status = 'PASS'
    elif actual <= threshold:
        status = 'WARN'
    else:
        status = 'FAIL'
        
    RESULTS.append({
        'check_name': name,
        'layer': layer,
        'actual': actual,
        'threshold': threshold,
        'status': status,
        'note': note
    })
    return status

# 1. Primary Key Uniqueness
run_check(
    'claims.claim_id_unique',
    "SELECT COUNT(*) FROM (SELECT claim_id FROM silver.claims GROUP BY 1 HAVING COUNT(*) > 1)",
    0, 'silver', 'One row per claim in published silver table'
)

# 2. Referential Integrity (Foreign Keys)
run_check(
    'claims.member_fk_valid',
    "SELECT COUNT(*) FROM silver.claims c LEFT JOIN silver.members m USING (member_id) WHERE m.member_id IS NULL",
    0, 'silver', 'Every claim must resolve to an active member'
)
run_check(
    'claims.provider_fk_valid',
    "SELECT COUNT(*) FROM silver.claims c LEFT JOIN silver.providers p USING (provider_id) WHERE p.provider_id IS NULL",
    0, 'silver', 'Every claim must resolve to an enrolled provider'
)
run_check(
    'claim_services.claim_fk_valid',
    "SELECT COUNT(*) FROM silver.claim_services s LEFT JOIN silver.claims c USING (claim_id) WHERE c.claim_id IS NULL",
    0, 'silver', 'Orphan service lines must never be published'
)

# 3. Critical Column Null Rates & Domain Constraints
run_check(
    'claims.billed_amount_null_pct',
    "SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE billed_amount IS NULL) / COUNT(*), 2) FROM silver.claims",
    1.0, 'silver', 'Null percentage on billed_amount (1% max tolerance)'
)
run_check(
    'raw.claims_status_in_domain',
    "SELECT COUNT(*) FROM raw.claims_api_records WHERE status NOT IN ('Submitted', 'Approved', 'Denied')",
    0, 'raw', 'As-landed statuses outside domain (mixed-case / invalid)'
)

# 4. Idempotency & Freshness
run_check(
    'bronze.claims_duplicate_claim_ids',
    "SELECT COUNT(*) FROM (SELECT claim_id FROM bronze.claims GROUP BY 1 HAVING COUNT(*) > 1)",
    0, 'bronze', 'Duplicate claim_ids indicating unmitigated replay in Bronze'
)
run_check(
    'freshness.bronze_claims_newest_load',
    "SELECT CASE WHEN date_diff('hour', MAX(_loaded_at), now()) > 24 THEN 1 ELSE 0 END FROM bronze.claims",
    0, 'bronze', 'Newest ingested batch must be less than 24 hours old'
)

scorecard = pd.DataFrame(RESULTS)
print(scorecard.to_string(index=False, max_colwidth=45))
print()
print(f"Scorecard Summary: {scorecard['status'].value_counts().to_dict()}")


In [ ]:
# Age of Bronze Rows by Ingestion Batch
print("Age of Bronze Ingestion Batches:")
print(q("""
SELECT 
    _ingest_batch, 
    COUNT(*) AS rows_loaded, 
    MIN(_loaded_at) AS loaded_at,
    date_diff('hour', MIN(_loaded_at), now()) AS age_hours
FROM bronze.claims 
GROUP BY 1 
ORDER BY loaded_at DESC
""").to_string(index=False))


---
### Drill 2: Cross-Validation with the Warehouse Reference Scorecard

**Business Question:** Does our custom framework align with the automated production DQ scorecard (`dq.dq_results` and `dq.dq_metrics`)?
* **The 1 `FAIL`:** `dashboard.reconciles_to_source` (99.73% off, threshold 1.0%). The published dashboard severely conflicts with `silver.claims`.
* **The 7 `WARN`s:** Explainable business edge-cases (reversals with negative amounts, paid > billed, timing differences).


In [ ]:
# Query Reference DQ Results and Metrics
print("Reference Scorecard Status Counts:")
print(q("""
SELECT status, COUNT(*) AS checks 
FROM dq.dq_results 
GROUP BY 1 
ORDER BY 1
""").to_string(index=False))

print()
print("Non-PASS Scorecard Items (Investigate First):")
print(q("""
SELECT check_name, layer, severity, actual, threshold, status
FROM dq.dq_results 
WHERE status <> 'PASS' 
ORDER BY status, actual DESC
""").to_string(index=False, max_colwidth=42))

print()
print("Warehouse DQ Health Metrics:")
print(q("""
SELECT metric, value 
FROM dq.dq_metrics 
ORDER BY metric
""").to_string(index=False, max_colwidth=48))


---
### Drill 3: Quarantine & Defect Distribution Analysis

**The Golden Rule:** Never drop a record silently. If a record fails schema validation or referential integrity, route it to `dq.quarantine` with the exact defect reason.
* Inspect `dq.quarantine` to identify top defect drivers across pipeline sources.
* Track quarantine volume drift over time.


In [ ]:
# Quarantine Profile by Source and Defect Reason
print("Quarantine Distribution by Defect Reason:")
print(q("""
SELECT 
    source, 
    reason, 
    COUNT(*) AS records,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dq.quarantine), 1) AS pct_of_quarantine
FROM dq.quarantine 
GROUP BY 1, 2 
ORDER BY records DESC
""").to_string(index=False, max_colwidth=48))

print()
print("Quarantine Timeline by Source System:")
print(q("""
SELECT 
    source, 
    COUNT(*) AS records, 
    MIN(quarantined_at) AS first_seen, 
    MAX(quarantined_at) AS last_seen
FROM dq.quarantine 
GROUP BY 1 
ORDER BY records DESC
""").to_string(index=False))


---
### Drill 4: Automated Null & Duplicate Profiling Across Silver Models

**Goal:** Dynamically profile key completeness, value null rates, and duplicate grains across all dimensional models without manual repetitive SQL.


In [ ]:
# Automated Dynamic Profiling Matrix
TABLES_TO_PROFILE = [
    ('silver.claims', ['claim_id'], 'paid_amount'),
    ('silver.claim_services', ['claim_id', 'service_id'], 'service_amount'),
    ('silver.payments', ['payment_id'], 'payment_amount'),
    ('silver.members', ['member_id'], 'plan_id'),
]

profile_results = []
for table, keys, value_col in TABLES_TO_PROFILE:
    total_rows = q1(f"SELECT COUNT(*) FROM {table}")
    distinct_keys = q1(f"SELECT COUNT(DISTINCT ({', '.join(keys)})) FROM {table}")
    any_key_null = ' OR '.join(f"{k} IS NULL" for k in keys)
    
    stats = q(f"""
        SELECT 
            COUNT(*) AS rows_total,
            ROUND(100.0 * COUNT(*) FILTER (WHERE {any_key_null}) / COUNT(*), 2) AS null_pct_keys,
            ROUND(100.0 * COUNT(*) FILTER (WHERE {value_col} IS NULL) / COUNT(*), 2) AS null_pct_value
        FROM {table}
    """).iloc[0].to_dict()
    
    stats['table'] = table
    stats['key_columns'] = ' + '.join(keys)
    stats['duplicate_pct'] = round(100.0 * (total_rows - distinct_keys) / total_rows, 2)
    profile_results.append(stats)

df_profile = pd.DataFrame(profile_results)[[
    'table', 'key_columns', 'rows_total', 'null_pct_keys', 'null_pct_value', 'duplicate_pct'
]]
print(df_profile.to_string(index=False))
print()
print("Profiled metric columns: paid_amount / service_amount / payment_amount / plan_id")


---
### Drill 5: PySpark Native Data Quality Assertions

**Senior DE Interview Question:** *"How do you implement data quality checks natively inside a distributed PySpark pipeline before writing to Delta lake?"*
* **Pattern 1 (PK Uniqueness):** Group by primary key and filter `count > 1`.
* **Pattern 2 (Referential Integrity):** Perform `left_anti` join to catch orphan records in distributed memory.
* **Pattern 3 (Null Rate Check):** Compute conditional null aggregates using `F.count(F.when(...))`.


In [ ]:
# PySpark Data Quality Implementation
is_remote = spark.__module__.startswith("pyspark.sql.connect") or hasattr(spark, "client")

# Load samples into Spark DataFrames
df_claims_spark = spark.createDataFrame(q("SELECT * FROM silver.claims").head(1000))
df_members_spark = spark.createDataFrame(q("SELECT * FROM silver.members"))

# 1. Spark PK Uniqueness Check
dup_count = (df_claims_spark
             .groupBy("claim_id")
             .count()
             .filter(F.col("count") > 1)
             .count())
print(f"PySpark Assert Unique PK (claim_id): {dup_count} duplicate keys -> {'PASS' if dup_count == 0 else 'FAIL'}")

# 2. Spark Orphan FK Check (Left Anti Join)
orphan_members = df_claims_spark.join(df_members_spark, "member_id", "left_anti")
orphan_count = orphan_members.count()
print(f"PySpark Assert FK Integrity (members): {orphan_count} orphan claims -> {'PASS' if orphan_count == 0 else 'FAIL'}")

# 3. Spark Null Rate Calculation
null_summary = df_claims_spark.select(
    F.count("*").alias("total"),
    F.count(F.when(F.col("billed_amount").isNull(), 1)).alias("billed_nulls"),
    F.count(F.when(F.col("paid_amount").isNull(), 1)).alias("paid_nulls")
).withColumn("paid_null_pct", F.round(100.0 * F.col("paid_nulls") / F.col("total"), 2))

null_summary.show(truncate=False)


---
### Drill 6: THE INVESTIGATION: "Dashboard says claims are up +40% MoM, but source says +5%"

**The Scenario:**
Executive leadership notices that the claims volume chart jumped **+40.1%** between March 2025 and April 2025 (`dashboard.monthly_claim_counts`).
However, the transactional source system reports steady growth of only **+5.0%**.

**The Systematic 6-Step Forensics Walkthrough:**
* **Step 1:** Compare row counts across all 5 Medallion stages.
* **Step 2:** Investigate duplicate loads (the April replay batch).
* **Step 3:** Check for join grain fan-out in dashboard data marts.
* **Step 4:** Trace silent drops and dropped records between Raw and Silver.
* **Step 5:** Detect date-boundary shifts (`service_month` vs `claim_date`).
* **Step 6:** Quantify the discrepancy and isolate root causes.


In [ ]:
# Step 1: Stage-by-Stage Row Counts
print("Step 1: Row Counts Across All Medallion Stages:")
stages = [
    ('raw.claims_api_records', 'SELECT COUNT(*) FROM raw.claims_api_records'),
    ('bronze.claims', 'SELECT COUNT(*) FROM bronze.claims'),
    ('silver.claims', 'SELECT COUNT(*) FROM silver.claims'),
    ('dashboard.claims_dashboard (rows)', 'SELECT COUNT(*) FROM dashboard.claims_dashboard'),
    ('dashboard.claims_dashboard (distinct claims)', 'SELECT COUNT(DISTINCT claim_id) FROM dashboard.claims_dashboard'),
]
for label, sql_expr in stages:
    count_val = q1(sql_expr)
    print(f"  {label:>46} : {count_val:>7,}")


In [ ]:
# Step 2: Uncovering Duplicate Loads in Bronze
print("Step 2: Identifying Ingestion Batches with Duplicate Claims:")
print(q("""
SELECT 
    _ingest_batch, 
    COUNT(*) AS rows_loaded, 
    COUNT(DISTINCT claim_id) AS distinct_claims,
    COUNT(*) - COUNT(DISTINCT claim_id) AS surplus_rows
FROM bronze.claims 
GROUP BY 1 
ORDER BY rows_loaded DESC
""").to_string(index=False))

print()
print("Batch Duplicate Breakdown:")
print(q("""
SELECT _ingest_batch, COUNT(*) AS duplicated_claim_ids
FROM (SELECT claim_id, _ingest_batch FROM bronze.claims GROUP BY 1, 2 HAVING COUNT(*) > 1)
GROUP BY 1 
ORDER BY duplicated_claim_ids DESC
""").to_string(index=False))
print()
print("Root Cause A: Batch '2025-04-16T02:05:00Z' was an unmitigated re-run (302 duplicate April claims loaded into Bronze).")


In [ ]:
# Step 3: Detecting Join Multiplication / Grain Fan-Out
print("Step 3: Checking Grain of Dashboard Mart:")
print(q("""
SELECT 
    COUNT(*) AS dashboard_rows, 
    COUNT(DISTINCT claim_id) AS distinct_claims,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT claim_id), 2) AS rows_per_claim
FROM dashboard.claims_dashboard
""").to_string(index=False))

print()
print("Comparing to Silver Claim Services Grain:")
print(q("""
SELECT 
    COUNT(*) AS service_lines, 
    COUNT(DISTINCT claim_id) AS claims,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT claim_id), 2) AS rows_per_claim
FROM silver.claim_services
""").to_string(index=False))
print()
print("Root Cause B: 'claims_dashboard' is at SERVICE-LINE grain (~3.3 lines per claim). COUNT(*) counts lines, not claims!")


In [ ]:
# Step 4: Tracking Silent Drops Between Raw and Silver
print("Step 4: Records That Never Reached Silver:")
print(q("""
WITH source AS (SELECT DISTINCT claim_id, member_id, provider_id, claim_date FROM raw.claims_api_records)
SELECT 
    COUNT(*) AS source_claim_ids,
    COUNT(*) FILTER (WHERE s.claim_id IS NULL) AS dropped_before_publish,
    COUNT(*) FILTER (WHERE s.claim_id IS NULL AND r.claim_id IS NULL) AS missing_claim_id,
    COUNT(*) FILTER (WHERE s.claim_id IS NULL AND r.claim_id IS NOT NULL AND r.member_id IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM silver.members m WHERE m.member_id = r.member_id)) AS member_fk_failed,
    COUNT(*) FILTER (WHERE s.claim_id IS NULL AND r.claim_id IS NOT NULL
          AND (r.member_id IS NULL OR EXISTS (SELECT 1 FROM silver.members m WHERE m.member_id = r.member_id))) AS other_malformed
FROM source r 
LEFT JOIN silver.claims s USING (claim_id)
""").to_string(index=False))

print()
print("Quarantine Breakdown for API Ingestion:")
print(q("""
SELECT reason, COUNT(*) AS records 
FROM dq.quarantine 
WHERE source = 'claims_api'
GROUP BY 1 
ORDER BY records DESC
""").to_string(index=False))


In [ ]:
# Step 5: Date-Boundary Shifts (Grouping Column Mismatch)
print("Step 5: Examining Month Allocation in Dashboard Mart:")
print(q("""
SELECT 
    service_month AS month, 
    COUNT(*) AS dashboard_rows,
    COUNT(*) FILTER (WHERE claim_month = service_month) AS same_claim_month,
    COUNT(*) FILTER (WHERE claim_month <> service_month) AS shifted_across_month_end
FROM dashboard.claims_dashboard
WHERE service_month IN (DATE '2025-03-01', DATE '2025-04-01')
GROUP BY 1 
ORDER BY 1
""").to_string(index=False))

print()
print("Sample Cross-Month Shifts:")
print(q("""
SELECT service_month, claim_month, COUNT(*) AS dashboard_rows
FROM dashboard.claims_dashboard
WHERE service_month <> claim_month AND service_month >= DATE '2025-03-01'
GROUP BY 1, 2 
ORDER BY dashboard_rows DESC 
LIMIT 5
""").to_string(index=False))
print()
print("Root Cause C: The dashboard groups by 'service_month', shifting 106 March claim lines into April!")


In [ ]:
# Step 6: Quantifying the Discrepancy
dashboard = q("""
SELECT month, dashboard_count, distinct_claims_in_dashboard 
FROM dashboard.monthly_claim_counts
WHERE month IN (DATE '2025-03-01', DATE '2025-04-01') 
ORDER BY month
""")

source = q("""
SELECT date_trunc('month', claim_date) AS month, COUNT(*) AS source_count 
FROM silver.claims
WHERE claim_date >= DATE '2025-03-01' AND claim_date < DATE '2025-05-01' 
GROUP BY 1 
ORDER BY 1
""")

print("Dashboard Reported Numbers:")
print(dashboard.to_string(index=False))
print()
print("Source True Numbers:")
print(source.to_string(index=False))

dash_diff = dashboard['dashboard_count'].iloc[1] - dashboard['dashboard_count'].iloc[0]
dash_pct = (dashboard['dashboard_count'].iloc[1] / dashboard['dashboard_count'].iloc[0] - 1) * 100

src_diff = source['source_count'].iloc[1] - source['source_count'].iloc[0]
src_pct = (source['source_count'].iloc[1] / source['source_count'].iloc[0] - 1) * 100

print()
print(f"Dashboard MoM: {dash_diff:+d} rows ({dash_pct:+.1f}%) -> ARTIFICIAL INFLATION")
print(f"Source MoM:    {src_diff:+d} claims ({src_pct:+.1f}%) -> TRUE REALITY")


---
### Drill 7: The Corrected Production Query

**The Solution:** Correct all four defects simultaneously:
1. Publish from **`silver.claims`** (deduplicated by business key).
2. Count **`DISTINCT claim_id`** (enforcing claim grain rather than service-line grain).
3. Group strictly by **`claim_date`** (preventing cross-month service date shifts).


In [ ]:
# YOUR TURN:
# Write the corrected monthly claim count from silver.
# Expected columns: ['claim_month', 'claims']


In [ ]:
# SOLUTION - Drill 7: Corrected Monthly Query
corrected = q("""
SELECT 
    date_trunc('month', c.claim_date) AS claim_month, 
    COUNT(DISTINCT c.claim_id) AS claims
FROM silver.claims c
JOIN silver.claim_services s USING (claim_id)
WHERE c.claim_date >= DATE '2025-01-01'
GROUP BY 1 
ORDER BY 1
""")

print("Corrected Monthly Trend (2025):")
print(corrected.to_string(index=False))

c_diff = corrected['claims'].iloc[3] - corrected['claims'].iloc[2]
c_pct = (corrected['claims'].iloc[3] / corrected['claims'].iloc[2] - 1) * 100
print()
print(f"True MoM Growth (March -> April): {c_diff:+d} claims ({c_pct:+.1f}%) -> Perfectly matches source system!")


---
### Drill 8: Senior DE Reconciliation Queries (Interview Style)

In data engineering system design interviews, you will often be asked: *"How do you write reconciliation queries to prove source and target data match?"*
* **Pattern 1:** Row count & distinct key reconciliation (`UNION ALL`).
* **Pattern 2:** Missing entity discovery (`LEFT JOIN ... WHERE target.id IS NULL`).
* **Pattern 3:** Duplicate key localization (`GROUP BY ... HAVING COUNT(*) > 1`).
* **Pattern 4:** Financial dollar balance reconciliation (comparing like-for-like, excluding voided payments).
* **Pattern 5:** Monthly variance localization (pinpointing the exact calendar month of divergence).


In [ ]:
# Recon 1: Source vs Target Row & Grain Count
print("Reconciliation 1: Source vs Target Grain Counts:")
print(q("""
SELECT 'all rows' AS grain, (SELECT COUNT(*) FROM raw.claims_api_records) AS source_count,
       (SELECT COUNT(*) FROM silver.claims) AS target_count
UNION ALL
SELECT 'distinct claim_id', (SELECT COUNT(DISTINCT claim_id) FROM raw.claims_api_records),
       (SELECT COUNT(DISTINCT claim_id) FROM silver.claims)
""").to_string(index=False))
print("Audit: 12,061 records -> 12,000 distinct claim_ids (60 revisions + 1 null id) -> 11,981 published (20 quarantined/dropped).")

print()
print("Reconciliation 2: Identifying Dropped Claims (LEFT JOIN ... IS NULL):")
print(q("""
SELECT r.claim_id, r.member_id, r.provider_id, r.claim_date
FROM (SELECT DISTINCT claim_id, member_id, provider_id, claim_date FROM raw.claims_api_records) r
LEFT JOIN silver.claims c USING (claim_id)
WHERE c.claim_id IS NULL 
ORDER BY r.claim_date NULLS LAST 
LIMIT 6
""").to_string(index=False))
total_missing = q1("""
SELECT COUNT(*) 
FROM (SELECT DISTINCT claim_id FROM raw.claims_api_records) r
LEFT JOIN silver.claims c USING (claim_id) 
WHERE c.claim_id IS NULL
""")
print(f"Total un-promoted source claims: {total_missing}")

print()
print("Reconciliation 3: Bronze Multi-Version Replay Detection:")
print(q("""
SELECT claim_id, COUNT(*) AS versions, MIN(updated_at) AS first_version, MAX(updated_at) AS last_version
FROM bronze.claims 
GROUP BY 1 
HAVING COUNT(*) > 1 
ORDER BY versions DESC, claim_id 
LIMIT 5
""").to_string(index=False))
dup_claims = q1("SELECT COUNT(*) FROM (SELECT claim_id FROM bronze.claims GROUP BY 1 HAVING COUNT(*) > 1)")
print(f"Bronze claims with multiple versions: {dup_claims}")


In [ ]:
# Recon 4: Financial Dollar Balance Reconciliation
print("Reconciliation 4: Claims Paid vs Payments Settled:")
print(q("""
SELECT 
    (SELECT SUM(paid_amount) FROM silver.claims) AS claims_paid_amount,
    (SELECT SUM(payment_amount) FROM silver.payments WHERE payment_status = 'Paid') AS payments_settled,
    (SELECT SUM(payment_amount) FROM silver.payments WHERE payment_status = 'Voided') AS payments_voided_excluded,
    (SELECT SUM(paid_amount) FROM silver.claims) - 
    (SELECT SUM(payment_amount) FROM silver.payments WHERE payment_status = 'Paid') AS variance,
    (SELECT COUNT(*) FROM (
        SELECT c.claim_id, c.paid_amount 
        FROM silver.claims c
        LEFT JOIN silver.payments p ON p.claim_id = c.claim_id AND p.payment_status = 'Paid'
        WHERE c.paid_amount IS NOT NULL AND c.paid_amount > 0
        GROUP BY 1, 2 
        HAVING ABS(c.paid_amount - COALESCE(SUM(p.payment_amount), 0)) > 0.01
    )) AS unmatched_paid_claims
""").to_string(index=False))

print()
print("Reconciliation 5: Localizing Payment Breaks by Month:")
print(q("""
WITH joined AS (
    SELECT c.claim_id, c.claim_date, c.paid_amount, COUNT(p.payment_id) AS payment_rows
    FROM silver.claims c 
    LEFT JOIN silver.payments p USING (claim_id) 
    GROUP BY 1, 2, 3
)
SELECT 
    date_trunc('month', claim_date) AS month, 
    COUNT(*) AS claims,
    COUNT(*) FILTER (WHERE paid_amount > 0) AS paid_claims,
    COUNT(*) FILTER (WHERE paid_amount > 0 AND payment_rows = 0) AS paid_without_payment_row,
    ROUND(SUM(paid_amount), 2) AS claims_paid
FROM joined 
WHERE claim_date >= DATE '2025-01-01' 
GROUP BY 1 
ORDER BY 1
""").to_string(index=False))


---
### The 5-Step Senior DE Incident Response Playbook

When an interviewer asks: *"The business reports that a metric in their BI dashboard does not match upstream systems. How do you triage, debug, and resolve the issue?"*

1. **Verify Metric Grain & Definitions:**
   - Confirm if the metric measures entity count (`claims`) or event child items (`service lines`).
   - Check if date dimensions match (e.g. `service_date` vs `claim_date` vs `payment_date`).
2. **Localize the Variance:**
   - Run slice-and-dice queries across dimensions (e.g. by month, source system, or state) to isolate where the difference originates instead of checking millions of rows globally.
3. **Trace Medallion Boundaries:**
   - Run row count and distinct key reconciliations at each hop: `Landing` $\to$ `Bronze` $\to$ `Silver` $\to$ `Gold Mart`.
   - Check quarantine tables (`dq.quarantine`) for unpromoted or dropped records.
4. **Inspect Pipeline Idempotency:**
   - Check for duplicate replay batches (`GROUP BY claim_id HAVING COUNT(*) > 1`) caused by blind append operations.
5. **Implement Permanent Guardrails:**
   - Enforce primary key uniqueness constraints on Silver/Gold models.
   - Add automated reconciliation assertions into CI/CD and daily scheduled pipeline runs, alerting on threshold breaches.
